# Clean Test Notebook
Backtest + Strategy Filter + ML Predictions

In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np
import joblib


In [ ]:

df = yf.download("^NSEBANK", interval="1m", period="5d")
df.dropna(inplace=True)


In [ ]:

df["ret"] = df["Close"].pct_change()
df["vol"] = df["Volume"].pct_change()
df.dropna(inplace=True)

X = df[["ret", "vol"]]


In [ ]:

model = joblib.load("model.pkl")

df["pred_prob"] = model.predict_proba(X)[:, 1]
df["pred_prob"] = df["pred_prob"].shift(1)
df.dropna(inplace=True)


In [ ]:

def add_strategy_context(df):
    df = df.copy()

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["rsi"] = 100 - (100 / (1 + rs))

    df["ema9"] = df["Close"].ewm(span=9).mean()
    df["ema21"] = df["Close"].ewm(span=21).mean()

    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    df["vwap"] = (tp * df["Volume"]).cumsum() / df["Volume"].cumsum()

    return df

df = add_strategy_context(df)
df.dropna(inplace=True)


In [ ]:

def strategy_filter(row, prob_th=0.55):
    if row["pred_prob"] < prob_th:
        return 0
    if row["ema9"] > row["ema21"] and row["rsi"] < 70 and row["Close"] > row["vwap"]:
        return 1
    return 0

df["signal"] = df.apply(strategy_filter, axis=1)


In [ ]:

capital = 100000
position = 0
entry_price = 0
trades = []

for i in range(1, len(df)):
    row = df.iloc[i]

    if position == 0 and row["signal"] == 1:
        position = 1
        entry_price = row["Open"]
        entry_time = row.name

    elif position == 1:
        if row["ema9"] < row["ema21"] or row["rsi"] > 75:
            exit_price = row["Open"]
            ret = (exit_price - entry_price) / entry_price

            trades.append({
                "entry_time": entry_time,
                "exit_time": row.name,
                "return": ret
            })
            position = 0


In [ ]:

trades_df = pd.DataFrame(trades)

print("Trades:", len(trades_df))
print("Win rate:", (trades_df["return"] > 0).mean())
print("Avg return:", trades_df["return"].mean())
print("Cumulative return:", (1 + trades_df["return"]).prod() - 1)
